## 13/05/2026

## A complete END to END Project

In [1]:
# pip install langchain langchain-community  langchain-ollama faiss-cpu pymupdf

In [2]:
# import subprocess, sys
# packages = [
#     "langchain-ollama", "langchain-community",
#     "langchain-classic", "langchain-core",
#     "langchain", "faiss-cpu", "pymupdf",
#     "docx2txt", "langchain-text-splitters"
# ]
# for pkg in packages:
#     subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"])
#     print(f"✅ {pkg}")

In [3]:
import subprocess
subprocess.run(["pip", "install", "langchain-classic", "-q"])
print("Done!")

Done!


**Meaning of why we used it:-langchain-classic package install karta hai.<br> 
Yeh isliye chahiye kyunki ConversationalRetrievalChain naye LangChain version,mein remove ho gayi — classic package mein abhi bhi available hai.**

In [4]:
# import subprocess

# result = subprocess.run(
#     ["ollama", "pull", "llama3.2:1b"],
#     capture_output=True, text=True
# )
# print(result.stdout)
# print(result.stderr)

----

In [5]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import ConversationalRetrievalChain   
from langchain_classic.memory import ConversationBufferMemory       
from langchain_core.prompts import PromptTemplate
import re

print("All imports done!")

All imports done!


**Kya karta hai: Sabhi zaruri tools import karta hai. Jaise ghar mein kaam karne se pehle sare tools table par rakh lo <br>
    — hammer, screwdriver etc. Yahan:<br>

ChatOllama = local LLM (llama3.2)<br>
OllamaEmbeddings = text ko numbers mein convert karta hai<br>
PyMuPDFLoader = PDF padhta hai <br>
FAISS = vector database <br>
ConversationalRetrievalChain = question-answer chain <br>
ConversationBufferMemory = chat history yaad rakhta hai <br>
PromptTemplate = LLM ko instructions deta hai** 

---

In [6]:
loader = PyMuPDFLoader("..\Data\Sonu_Jha_Resume.pdf")
docs = loader.load()

print(f"Pages loaded: {len(docs)}")
print(f"Type: {type(docs[0])}")
print(docs[0].page_content[:200])

Pages loaded: 1
Type: <class 'langchain_core.documents.base.Document'>
Sonu Kumar Jha 
Gurugram, Haryana, India | sonuk020658@gmail.com | 8595302445 
https://www.linkedin.com/in/sonu-jha-b69b4b299/ | https://github.com/Sonu-jha69 
Summary 
Data Science Enthusiast with a 


**Kya karta hai: PDF file kholta hai aur text extract karta hai. <br>
Jaise ek scanner jo PDF ko padhkar text nikalta hai. Output ek Document object hai jisme .<br>
page_content (text) aur .metadata (file info) hota hai.**

----

In [7]:
for doc in docs:
    text = doc.page_content
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    doc.page_content = text.strip()

print("Cleaned!")
print(docs[0].page_content[:300])

Cleaned!
Sonu Kumar Jha Gurugram, Haryana, India | sonuk020658@gmail.com | 8595302445 https://www.linkedin.com/in/sonu-jha-b69b4b299/ | https://github.com/Sonu-jha69 Summary Data Science Enthusiast with a strong foundation in analytics, machine learning, and problem-solving. Demonstrated leadership through a


**Kya karta hai: PDF se aaye gande characters saaf karta hai. PDF mein special symbols, extra spaces, line breaks hote hain — yeh sab remove ho<br> jaate hain. Jaise kapde dhone se daag saaf hote hain.<br>
Line 1 = non-English/special characters remove karo<br>
Line 2 = extra spaces ek space mein badlo.**

----

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n=== Chunk {i+1} ===")
    print(chunk.page_content)

Total chunks: 7

=== Chunk 1 ===
Sonu Kumar Jha Gurugram, Haryana, India | sonuk020658@gmail.com | 8595302445 https://www.linkedin.com/in/sonu-jha-b69b4b299/ | https://github.com/Sonu-jha69 Summary Data Science Enthusiast with a strong foundation in analytics, machine learning, and problem-solving. Demonstrated leadership through academic and extracurricular achievements, including experience in tech-driven projects and data-based decision-making. Adept at turning raw data into actionable insights and driving collaboration to

=== Chunk 2 ===
decision-making. Adept at turning raw data into actionable insights and driving collaboration to achieve impactful results. Technologies Programming Languages: Java, Python Technologies: React, Next Js, Django, CSS, IoT Development Practices: Data Structures and Algorithms, Object-Oriented Programming, software design Tools: Git/GitHub Experience Tech Support Intern, Tradebox Noida, India February 2024 April 2024 Developed Tradebox UI using Next.j

**Kya karta hai: Poora PDF text chhote-chhote pieces mein toot jaata hai. Ek badi kitaab ke pages ki tarah — <br>
ek baar mein poori kitaab nahi padhte, page by page padhte hain.<br>
chunk_size=500 = har piece mein max 500 characters<br>
chunk_overlap=100 = do chunks ke beech 100 characters common rahenge — taaki context toot na jaye<br>
Result = 11 chunks bane**

----

In [9]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print("FAISS created!")

FAISS created!


**Kya karta hai: Teen kaam:<br>
Har chunk ka meaning numbers (vectors) mein convert hota hai — jaise "Rahul is a Data Scientist" → [0.23, 0.87, 0.12, ...]<br>
Yeh sare numbers FAISS database mein store hote hain<br>
Database disk pe save hota hai — taaki dobara embed na karna pade**


----

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 11})

query = "Who is Sonu Kumar Jha?"
retrieved = retriever.invoke(query)

for i, doc in enumerate(retrieved):
    print(f"\n=== Chunk {i+1} ===")
    print(doc.page_content)


=== Chunk 1 ===
decision-making. Adept at turning raw data into actionable insights and driving collaboration to achieve impactful results. Technologies Programming Languages: Java, Python Technologies: React, Next Js, Django, CSS, IoT Development Practices: Data Structures and Algorithms, Object-Oriented Programming, software design Tools: Git/GitHub Experience Tech Support Intern, Tradebox Noida, India February 2024 April 2024 Developed Tradebox UI using Next.js, increasing customer acquisition by 40% and

=== Chunk 2 ===
Sonu Kumar Jha Gurugram, Haryana, India | sonuk020658@gmail.com | 8595302445 https://www.linkedin.com/in/sonu-jha-b69b4b299/ | https://github.com/Sonu-jha69 Summary Data Science Enthusiast with a strong foundation in analytics, machine learning, and problem-solving. Demonstrated leadership through academic and extracurricular achievements, including experience in tech-driven projects and data-based decision-making. Adept at turning raw data into actionable insights

**Kya karta hai: Ek search engine banata hai jo FAISS mein se relevant chunks dhundhta hai.<br>
 k=11 matlab — query ke liye top 11 matching chunks lao. Tumhare resume mein total 11 chunks hain isliye sab liye.<br>
Jaise Google search karta hai — query dalo, related results milte hain.**

------

In [11]:
# llm = ChatOllama(model="llama3.2", temperature=0,num_ctx=2048,num_gpu=1)
llm = ChatOllama(model="llama3.2", temperature=0,num_ctx=2048,num_gpu=1)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are reading a resume document.
The resume belongs to the person mentioned in it.
Extract and answer directly from the resume text given below.
Give a clear, specific answer. Do not say information is missing if it exists in the text.


Context:
{context}

Question: {question}
Answer:"""
)

C:\Users\Rahul Kumar Jha\AppData\Local\Temp\ipykernel_15208\1694755783.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


**Kya karta hai: Char cheezein ek saath:<br>
llm = Local AI model (llama3.2) jo answer generate karta hai. temperature=0 = creative nahi, sirf factual answers.<br>
memory = Pichle sawaal yaad rakhta hai — isliye follow-up questions kaam karte hain.<br>
custom_prompt = LLM ko instruction deta hai: "Sirf document ke context se answer do, bahar se kuch mat bolo."<br>
qa_chain = Sab kuch ek pipeline mein jodd deta hai: Question → Retriever → LLM → Answer.**

In [12]:
from langchain_core.messages import HumanMessage
res = llm.invoke([HumanMessage(content="What is 2+2?")])
print(res.content)

2 + 2 = 4.


In [13]:
# # Test karo
# from langchain_core.messages import HumanMessage
# res = llm.invoke([HumanMessage(content="What is 2+2?")])
# print(res.content)

In [14]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": custom_prompt},
    verbose=False
)

print("qa_chain ready!")

qa_chain ready!


-----

In [15]:
result = qa_chain.invoke({"question": "Who is Sonu Kumar Jha?"})
print("Answer:", result["answer"])

Answer: Sonu Kumar Jha is a Data Science Enthusiast.


**Kya karta hai: Final step — question pucho aur answer lo. Internally yeh hota hai:<br>
Question embed hota hai<br>
FAISS se 11 chunks milte hain<br>
Chunks + Question LLM ko diya jaata hai<br>
LLM answer generate karta hai**

----

In [16]:
# # Test 2
# result = qa_chain.invoke({"question": "What are his technical skills?"})
# print("Answer:", result["answer"])

In [17]:
# Test 3
result = qa_chain.invoke({"question": "What projects has he built?"})
print("Answer:", result["answer"])

Answer: Sonu Kumar Jha has built the following projects:

1. A live bus tracking system for 50 buses, enabling real-time monitoring for over 1,000 users.
2. An AI-powered driver drowsiness detection system that cut fatigue-related incidents by 30%.
3. Bus route optimization that reduced fuel consumption by 250 kg.
4. An Electronic Health Record System that enhanced record retrieval efficiency by 40%.
5. A centralized medical records repository.

Additionally, Sonu Kumar Jha has also worked on other projects such as:

1. Developed Tradebox UI using Next.js, increasing customer acquisition by 40% and enhancing user experience by 30%.
2. Created a reward points feature that boosted user engagement by 25%.
3. Collaborated with cross-functional teams to reduce feature rollout time by 20% through effective UI-backend integration.
4. Created the company website for Thermocraft Engineering Services, enhancing functionality, usability, and aesthetics, which increased user satisfaction by 30%.


In [18]:
# Test 3
result = qa_chain.invoke({"question": "What is the GPA?"})
print("Answer:", result["answer"])

Answer: Sonu Kumar Jha's GPA is 8.1.


In [19]:
import re

def print_answer(result):
    answer = result["answer"]
    
    # Split only where full stop is followed by capital letter (real sentence end)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', answer)
    
    print("Answer:\n")
    for sentence in sentences:
        sentence = sentence.strip()
        if sentence:
            print(sentence)
            print()

# Test
result = qa_chain.invoke({"question": "Tell me about his internship"})
print_answer(result)

Answer:

Sonu Kumar Jha's internships were with Tradebox Noida, India (Tech Support Intern) from February 2024 to April 2024, where he developed the Tradebox UI using Next.js, increasing customer acquisition by 40% and enhancing user experience by 30%.

Additionally, his internship was with Thermocraft Engineering Services Gurugram as a Software Engineer Intern from November 2023 to December 2023.



-----

In [20]:
## Streamlit Process :---

In [21]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "streamlit", "-q"])
print("Streamlit installed!")

Streamlit installed!


In [22]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "fpdf2", "-q"])
print("fpdf2 installed!")

fpdf2 installed!


In [23]:
import subprocess, sys

file_path = r"C:\Users\Rahul Kumar Jha\GenAI Project\Project\resume_chatbot.py"

subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", file_path,
    "--server.port", "8501",
    "--server.headless", "false"
])

import time
time.sleep(6)

print("✅ Streamlit started!")
print("👉 Open: http://localhost:8501")

✅ Streamlit started!
👉 Open: http://localhost:8501


In [24]:
# import subprocess
# result = subprocess.run(
#     ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
#      "--format=csv,noheader"],
#     capture_output=True, text=True
# )
# print(result.stdout)

------

In Below Code Means that I remove llama3.2:1b which i install because in previous time ,Our llama3.2 was giving answer current But Now I solve this this problems So i do not need llama3.2:1b

In [ ]:
# import subprocess
# result = subprocess.run(
#     ["ollama", "rm", "llama3.2:1b"],
#     capture_output=True, text=True
# )
# print(result.stdout)
# print("✅ llama3.2:1b deleted!")

deleted 'llama3.2:1b'

✅ llama3.2:1b deleted!


In [ ]:
# result = subprocess.run(
#     ["ollama", "list"],
#     capture_output=True, text=True
# )
# print(result.stdout)
# # Only llama3.2 should show now

NAME                       ID              SIZE      MODIFIED    
nomic-embed-text:latest    0a109f422b47    274 MB    6 days ago     
llama3.2:latest            a80c4f17acd5    2.0 GB    4 weeks ago    

